### IMPORT LIBRARIES

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.docstore.document import Document
from langchain_cohere import CohereEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_community.retrievers import BM25Retriever


loader = PyPDFLoader('D:\Hamza Ali\Hybrid-RAG-With-Cohere-Reranking\Swarm_Intelligence\Swarm_Intelligence Lecture_1.pdf')

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

URL = os.getenv("QDRANT_URL")
API_KEY = os.getenv("QDRANT_API_KEY")
COHERE_API_KEY = os.getenv("COHERE_API_KEY")

In [ ]:
new_pages = loader.load()

In [ ]:
pages = loader.load()

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=900,
    chunk_overlap=150
)

In [ ]:
doc_list = []
for page in new_pages:
    splits = text_splitter.split_text(page.page_content)
    for split in splits:
        doc_store = Document(page_content="*** LECTURE 1 CONTENT ***" + split +  "***page_number : ***" + str(page.metadata['page']))
        doc_list.append(doc_store)

In [ ]:
for page in pages:
    splits = text_splitter.split_text(page.page_content)
    for split in splits:
        doc_store = Document(page_content="*** LECTURE 2 CONTENT ***" + split +  "***page_number : ***" + str(page.metadata['page']))
        doc_list.append(doc_store)

In [ ]:
# Define the Cohere embedding model
embeddings = CohereEmbeddings(
    cohere_api_key=COHERE_API_KEY, model="embed-english-v3.0"
)

In [ ]:
vector_store = QdrantVectorStore.from_documents(
    documents=doc_list,
    embedding=embeddings,
    collection_name='Universty Lecture Notes',
    url=URL,
    api_key=API_KEY
)

22 OCT CODE NEW CODE

Vector Search

In [ ]:
vector_db = QdrantVectorStore.from_existing_collection(
    collection_name='Universty Lecture Notes',
    embedding=embeddings,
    url=URL,
    api_key=API_KEY,
    
)

BM25 SEARCH

In [ ]:
bm25_retriever = BM25Retriever.from_documents(doc_list)
bm25_retriever.k = 3

In [ ]:
query = "Honeybee foraging & the waggle dance"

In [ ]:
query_to_bm25 = bm25_retriever._get_relevant_documents(query="Honeybee foraging & the waggle dance",run_manager=doc_list)

In [ ]:
query_to_bm25 

VECTOR RETRIEVER

In [ ]:
vector_retriever = vector_db.as_retriever()

result = vector_retriever.invoke("Honeybee foraging & the waggle dance")
result